# Low-inlier rescue rule: does it hold in both domains?
Question from the gate work: raw inlier counts mean different things in official and NUDT, so which signals are domain-robust for answering photos with 3 to 7 inliers? Data: `gate_rows2_official` (379 photos, 100 to 351 m) and `gate_rows2_nudt` (300 photos, 205 m up), both 6-query pipeline, conf 0.22, from `Gate_Features_Run_v2.ipynb`.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
here = pathlib.Path.cwd()
repo = next(p for p in [here, *here.parents] if (p / "src").is_dir() and (p / "train" / "checks").is_dir())
sys.path[:0] = [str(repo), str(repo / "src")]
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from train.checks.decision_gate import points_from_error, score_of_flags, rescue_flags, paired_bootstrap_gain
R = repo / "tests/Test_Iteration_3/results/fresh_run/"
sources = {"official": pd.read_csv(R / "gate_rows2_official_conf0.22.csv"), "nudt": pd.read_csv(R / "gate_rows2_nudt_conf0.22.csv")}
for d in sources.values():
    d["points"] = points_from_error(d.error_sigma.values)
pd.DataFrame({k: dict(photos=len(d), bar8=score_of_flags(d.points.values, d.inliers.values >= 8)[0], always=score_of_flags(d.points.values, np.ones(len(d), bool))[0],
                      ceiling=score_of_flags(d.points.values, d.points.values > 0)[0]) for k, d in sources.items()}).T.round(2)

,photos,bar8,always,ceiling
official,379.0,49.01,54.17,57.02
nudt,300.0,53.00,44.68,54.68


## 1. Which signal separates right from wrong among answers with 3 to 7 inliers (AUC, 0.5 = useless)

In [2]:
rows = {}
for name, d in sources.items():
    low = d[(d.inliers >= 3) & (d.inliers < 8)]
    rows[name] = {f: roc_auc_score(low.points > 0, low[f]) for f in ["inliers", "z_chosen", "agree_queries", "peak_sharpness", "peak_score", "others_p90", "height"]}
    rows[name]["n / right"] = "%d / %d" % (len(low), (low.points > 0).sum())
pd.DataFrame(rows).T

,inliers,z_chosen,agree_queries,peak_sharpness,peak_score,others_p90,height,n / right
official,0.505319,0.946809,0.765957,0.797872,0.978723,0.5,0.601064,49 / 47
nudt,0.821429,0.901786,0.928571,0.785714,0.392857,0.46875,0.741071,23 / 7


Raw inliers separate in NUDT (0.82) but not in official (0.51); the coarse z-score and the query agreement separate in both (about 0.8 to 0.95). So a rule built on z and agreement is the logically consistent one. Competitor strength (others_p90) and peak score do not carry across.

In [3]:
settings = [(8, 99, 99), (4, 2.0, 2), (5, 2.0, 2), (3, 2.0, 2), (4, 2.0, 0), (4, 2.5, 2), (4, 3.0, 2)]
rows = []
for low_bar, z_min, agree_min in settings:
    row = dict(rule="low_bar %d, z>=%.1f, agree>=%d" % (low_bar, z_min, agree_min))
    for name, d in sources.items():
        old = d.inliers.values >= 8
        new = rescue_flags(d, 8, low_bar, z_min, agree_min)
        gain, low, high = paired_bootstrap_gain(d.points.values, new, old)
        row[name + " gain"] = round(gain, 2); row[name + " 95%"] = "[%.2f, %.2f]" % (low, high); row[name + " wrong"] = score_of_flags(d.points.values, new)[1]
    rows.append(row)
pd.DataFrame(rows)

,rule,official gain,official 95%,official wrong,nudt gain,nudt 95%,nudt wrong
0,"low_bar 8, z>=99.0, agree>=99",0.00,"[0.00, 0.00]",0,0.00,"[0.00, 0.00]",0
1,"low_bar 4, z>=2.0, agree>=2",2.31,"[1.27, 3.58]",0,0.92,"[0.20, 1.80]",0
2,"low_bar 5, z>=2.0, agree>=2",1.68,"[0.79, 2.79]",0,0.92,"[0.20, 1.80]",0
3,"low_bar 3, z>=2.0, agree>=2",2.31,"[1.27, 3.58]",0,0.92,"[0.20, 1.80]",0
4,"low_bar 4, z>=2.0, agree>=0",4.24,"[2.75, 5.89]",0,0.52,"[-0.64, 1.60]",1
5,"low_bar 4, z>=2.5, agree>=2",1.58,"[0.63, 2.53]",0,0.20,"[0.00, 0.60]",0
6,"low_bar 4, z>=3.0, agree>=2",0.32,"[0.00, 0.79]",0,0.20,"[0.00, 0.60]",0


## Reading
- One parameter set (answer at 4 to 7 inliers when z >= 2 and at least 2 of the 6 queries agree) gains in both domains with 0 wrong answers: about +2.3 marks on official, +0.9 on NUDT. It is a small gain, the interval on NUDT is wide, and the three settings were chosen after seeing both sets.
- Dropping the agreement condition gains more on official (+4.2) but adds a wrong answer on NUDT; dropping z and agreement together (answer at 4+ inliers) adds 13 wrong answers on NUDT. So both conditions earn their place.
- Not a result on new data: same photos as the dev sets. Treat as a candidate rule to confirm on Cartosat photos outside the dev set.